# Similarity Score Models
Three methods compared: User-User CF, Item-Item CF, Gradient Boosting hybrid.
Evaluation: Hit Rate@10, Recall@10, NDCG@10.

In [1]:
import ast
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import GradientBoostingClassifier

## 1. Load Data

In [2]:
import os
os.chdir('/Users/saurav/Documents/Spring-2026/CS_4641_Machine_Learning/movie-recommendation-system/data')

movies  = pd.read_csv('./movies_metadata.csv', low_memory=False)
keywords = pd.read_csv('./keywords.csv')
credits = pd.read_csv('./credits.csv')
ratings  = pd.read_csv('./ratings_small.csv')

# --- same cleaning as exploration notebook ---
movies['id']       = pd.to_numeric(movies['id'], errors='coerce')
ratings['movieId'] = pd.to_numeric(ratings['movieId'], errors='coerce')
movies  = movies.dropna(subset=['id']).copy()
ratings = ratings.dropna(subset=['movieId']).copy()
movies['id']       = movies['id'].astype('int64')
ratings['movieId'] = ratings['movieId'].astype('int64')

movies  = movies.drop_duplicates(subset='id')
ratings = ratings.dropna(subset=['rating'])

active_users = ratings['userId'].value_counts()
active_users = active_users[active_users >= 3].index
ratings = ratings[ratings['userId'].isin(active_users)]

# keep only movies that appear in ratings
rated_ids = set(ratings['movieId'].unique())
movies = movies[movies['id'].isin(rated_ids)]

print('Movies:', movies.shape, '  Ratings:', ratings.shape)

Movies: (2830, 24)   Ratings: (100004, 4)


## 2. Shared Structures

In [3]:
# User-item matrix (rows=users, cols=movies, values=ratings, NaN where unrated)
user_item = ratings.pivot_table(index='userId', columns='movieId', values='rating')
user_item_filled = user_item.fillna(0)   # zero-filled copy for dot products

all_movie_ids = set(user_item.columns)
all_user_ids  = list(user_item.index)

# Precompute user-user cosine similarity matrix (users x users)
user_sim_matrix = cosine_similarity(user_item_filled.values)   # shape (U, U)

print(f'User-item matrix: {user_item.shape}')
print(f'User similarity matrix: {user_sim_matrix.shape}')

User-item matrix: (671, 9066)
User similarity matrix: (671, 671)


## 3. Item-Item Feature Matrix

In [4]:
def parse_name_list(s):
    """Parse a stringified list-of-dicts and return space-joined names."""
    try:
        return ' '.join(d['name'] for d in ast.literal_eval(s))
    except Exception:
        return ''

def parse_credits(crew_str, cast_str):
    """Extract director from crew and top 3 billed actors from cast."""
    try:
        director = ' '.join(
            d['name'] for d in ast.literal_eval(crew_str)
            if d.get('job') == 'Director'
        )
    except Exception:
        director = ''
    try:
        top_cast = ' '.join(
            d['name'] for d in ast.literal_eval(cast_str)[:3]
        )
    except Exception:
        top_cast = ''
    return director + ' ' + top_cast

kw_lookup      = dict(zip(keywords['id'], keywords['keywords']))
credits['id']  = pd.to_numeric(credits['id'], errors='coerce').astype('Int64')
crew_lookup    = dict(zip(credits['id'], credits['crew']))
cast_lookup    = dict(zip(credits['id'], credits['cast']))

movies['genres_str']   = movies['genres'].apply(parse_name_list)
movies['keywords_str'] = movies['id'].apply(lambda mid: parse_name_list(kw_lookup.get(mid, '[]')))
movies['credits_str']  = movies['id'].apply(lambda mid: parse_credits(
    crew_lookup.get(mid, '[]'),
    cast_lookup.get(mid, '[]')
))

# CountVectorizer version (original)
movies['item_features'] = movies['genres_str'] + ' ' + movies['keywords_str']

# TF-IDF version (genres + keywords + credits)
movies['item_features_tfidf'] = movies['genres_str'] + ' ' + movies['keywords_str'] + ' ' + movies['credits_str']

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer       = CountVectorizer(stop_words='english')
item_feat_matrix = vectorizer.fit_transform(movies['item_features'])
item_sim_matrix  = cosine_similarity(item_feat_matrix)

vectorizer_tfidf       = TfidfVectorizer(stop_words='english')
item_feat_matrix_tfidf = vectorizer_tfidf.fit_transform(movies['item_features_tfidf'])
item_sim_matrix_tfidf  = cosine_similarity(item_feat_matrix_tfidf)

movie_id_to_idx = {mid: i for i, mid in enumerate(movies['id'])}

print(f'Item similarity matrix (Count):  {item_sim_matrix.shape}')
print(f'Item similarity matrix (TF-IDF): {item_sim_matrix_tfidf.shape}')

Item similarity matrix (Count):  (2830, 2830)
Item similarity matrix (TF-IDF): (2830, 2830)


## 4. Scoring Functions
Each function takes a `user_id` and a list of `candidate_movie_ids`
and returns a **numpy array of scores** in the same order.

In [5]:
TOP_N_USERS = 10   # neighbours for user-user CF
K = 3              # evaluation cutoff

user_idx_map = {uid: i for i, uid in enumerate(user_item_filled.index)}


def score_user_user(user_id, candidate_ids):
    """Average rating of the top-N similar users for each candidate movie."""
    u_idx = user_idx_map[user_id]
    sims  = user_sim_matrix[u_idx]                         # (U,)
    top_n_idx = np.argsort(sims)[::-1][1:TOP_N_USERS + 1] # exclude self
    neighbour_ratings = user_item_filled.values[top_n_idx][:, [user_item_filled.columns.get_loc(m) for m in candidate_ids]]
    return neighbour_ratings.mean(axis=0)


def score_item_item(user_id, candidate_ids):
    """For each candidate, average item-item similarity to the user's rated movies."""
    rated = ratings[ratings['userId'] == user_id]['movieId'].values
    rated_idxs = [movie_id_to_idx[m] for m in rated if m in movie_id_to_idx]
    scores = []
    for m in candidate_ids:
        idx = movie_id_to_idx.get(m)
        if idx is None or not rated_idxs:
            scores.append(0.0)
        else:
            scores.append(item_sim_matrix[idx, rated_idxs].mean())
    return np.array(scores)

def score_item_item_tfidf(user_id, candidate_ids):
    """Item-item similarity using TF-IDF vectors with genres, keywords, and credits."""
    rated = ratings[ratings['userId'] == user_id]['movieId'].values
    rated_idxs = [movie_id_to_idx[m] for m in rated if m in movie_id_to_idx]
    scores = []
    for m in candidate_ids:
        idx = movie_id_to_idx.get(m)
        if idx is None or not rated_idxs:
            scores.append(0.0)
        else:
            scores.append(item_sim_matrix_tfidf[idx, rated_idxs].mean())
    return np.array(scores)

## 5. Train Gradient Boosting Model
Train/test split is at the **user level** to avoid leakage.

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_CANDIDATES = 100   # 1 watched + 99 unrated

def build_sample(user_id):
    """Return (X_rows, y_rows, watched_movie) for one user, or None if skipped."""
    user_rated = ratings[ratings['userId'] == user_id]['movieId'].unique()
    not_watched = list(all_movie_ids - set(user_rated))
    if len(user_rated) == 0 or len(not_watched) < N_CANDIDATES - 1:
        return None

    watched_movie   = random.choice(user_rated)
    negative_sample = random.sample(not_watched, N_CANDIDATES - 1)
    candidates      = [watched_movie] + negative_sample
    random.shuffle(candidates)

    uu_scores = score_user_user(user_id, candidates)
    ii_scores = score_item_item_tfidf(user_id, candidates)

    X_rows = np.stack([uu_scores, ii_scores], axis=1)   # (N_CANDIDATES, 2)
    y_rows = np.array([int(m == watched_movie) for m in candidates])
    return X_rows, y_rows, watched_movie, candidates


# --- user-level train/test split ---
eligible_users = [uid for uid in all_user_ids]
random.shuffle(eligible_users)
split = int(0.8 * len(eligible_users))
train_users = eligible_users[:split]
test_users  = eligible_users[split:]

X_train, y_train = [], []
for uid in train_users:
    result = build_sample(uid)
    if result:
        X_train.append(result[0])
        y_train.append(result[1])

X_train = np.vstack(X_train)
y_train = np.concatenate(y_train)

gb_model = GradientBoostingClassifier(random_state=SEED)
gb_model.fit(X_train, y_train)
print('Gradient boosting model trained.')
print(f'  Train samples: {len(X_train)}  (pos={y_train.sum()}, neg={len(y_train)-y_train.sum()})')

Gradient boosting model trained.
  Train samples: 53600  (pos=536, neg=53064)


In [7]:
def score_gradient_boost(user_id, candidate_ids):
    """Use GB model's positive-class probability as the score."""
    uu = score_user_user(user_id, candidate_ids)
    ii = score_item_item(user_id, candidate_ids)
    X  = np.stack([uu, ii], axis=1)
    return gb_model.predict_proba(X)[:, 1]

## 6. Evaluation  —  Hit Rate@K, Recall@K, NDCG@K

In [8]:
def evaluate_models(user_user_cf, item_item_cf, gradient_boosting, users, k=K, n_candidates=N_CANDIDATES, seed=SEED):
    
    random.seed(seed)
    np.random.seed(seed)

    hits_uu, ndcgs_uu = [], [] 
    hits_ii, ndcgs_ii = [], []
    hits_gb, ndcgs_gb = [], []

    for user_id in users:
        user_rated  = ratings[ratings['userId'] == user_id]['movieId'].unique()
        not_watched = list(all_movie_ids - set(user_rated))
        if len(user_rated) == 0 or len(not_watched) < n_candidates - 1:
            continue

        watched_movie   = random.choice(user_rated)
        negative_sample = random.sample(not_watched, n_candidates - 1)
        candidates      = [watched_movie] + negative_sample
        random.shuffle(candidates)

        scores_uu     = user_user_cf(user_id, candidates)
        ranked_ids_uu = [candidates[i] for i in np.argsort(scores_uu)[::-1]]
        relevance_uu  = [int(m == watched_movie) for m in ranked_ids_uu]

        hits_uu.append(int(watched_movie in ranked_ids_uu[:k]))
        ndcg_score_uu = 1 / np.log2(relevance_uu.index(1) + 2) if 1 in relevance_uu else 0
        ndcgs_uu.append(ndcg_score_uu)

        scores_ii     = item_item_cf(user_id, candidates)
        ranked_ids_ii = [candidates[i] for i in np.argsort(scores_ii)[::-1]]
        relevance_ii  = [int(m == watched_movie) for m in ranked_ids_ii]
        hits_ii.append(int(watched_movie in ranked_ids_ii[:k]))
        ndcgs_ii.append(1 / np.log2(relevance_ii.index(1) + 2) if 1 in relevance_ii else 0)    

        scores_gb     = gradient_boosting(user_id, candidates)
        ranked_ids_gb = [candidates[i] for i in np.argsort(scores_gb)[::-1]]
        relevance_gb  = [int(m == watched_movie) for m in ranked_ids_gb]
        hits_gb.append(int(watched_movie in ranked_ids_gb[:k]))
        ndcg_score_gb = 1 / np.log2(relevance_gb.index(1) + 2) if 1 in relevance_gb else 0
        ndcgs_gb.append(ndcg_score_gb)

    return {
        'hit_rate_ii': np.mean(hits_ii),
        'ndcg_ii':     np.mean(ndcgs_ii),
        'hit_rate_uu': np.mean(hits_uu),
        'ndcg_uu':     np.mean(ndcgs_uu),
        'hit_rate_gb': np.mean(hits_gb),
        'ndcg_gb':     np.mean(ndcgs_gb),
        'hit_list_UU': hits_uu,
        'ndcg_list_UU': ndcgs_uu,
        'hit_list_II': hits_ii,
        'ndcg_list_II': ndcgs_ii,
        'hit_list_GB': hits_gb,
        'ndcg_list_GB': ndcgs_gb,
    }

## 7. Visualisations

In [11]:
print(f'Evaluating on {len(test_users)} held-out test users...\n')

results   = evaluate_models(score_user_user, score_item_item_tfidf, score_gradient_boost, test_users)

results = pd.DataFrame({
    'Model': ['User-User CF', 'Item-Item CF (TF-IDF)', 'Gradient Boosting'],
    'Hit Rate@10': [results['hit_rate_uu'], results['hit_rate_ii'], results['hit_rate_gb']],
    'NDCG@10': [results['ndcg_uu'], results['ndcg_ii'], results['ndcg_gb']]
})


print(results)

# # --- Bar chart ---
# models_plot = ['User-User CF', 'Item-Item\n(Count)', 'Item-Item\n(TF-IDF)', 'Gradient Boost']
# metrics     = ['Hit Rate@10', 'NDCG@10']
# x     = np.arange(len(metrics))
# width = 0.2

# fig, ax = plt.subplots(figsize=(10, 5))
# for i, (model, res) in enumerate(zip(models_plot, all_results)):
#     vals = [res['hit_rate'], res['ndcg']]
#     ax.bar(x + i * width, vals, width, label=model)

# ax.set_xticks(x + width * 1.5)
# ax.set_xticklabels(metrics)
# ax.set_ylim(0, 1)
# ax.set_ylabel('Score')
# ax.set_title('Model Comparison @ K=10')
# ax.legend()
# plt.tight_layout()
# plt.show()

# # --- Per-user NDCG distributions ---
# n = min(len(r['ndcg_list']) for r in all_results)

# fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
# for ax, label, res in zip(axes, models_plot, all_results):
#     ax.plot(range(n), res['ndcg_list'][:n], marker='o', markersize=3, linewidth=0.8)
#     ax.axhline(res['ndcg'], color='red', linestyle='--', label=f"mean={res['ndcg']:.2f}")
#     ax.set_title(label.replace('\n', ' '))
#     ax.set_xlabel('Test User Index')
#     ax.set_ylabel('NDCG@10')
#     ax.set_ylim(-0.05, 1.1)
#     ax.legend(fontsize=8)

# plt.suptitle('Per-User NDCG@10 Across Test Users', y=1.02)
# plt.tight_layout()
# plt.show()

Evaluating on 135 held-out test users...

                   Model  Hit Rate@10   NDCG@10
0           User-User CF     0.607407  0.624095
1  Item-Item CF (TF-IDF)     0.474074  0.536143
2      Gradient Boosting     0.259259  0.380170


In [16]:
import ast
import numpy as np
import pandas as pd
import random

def parse_genres(x):
    try:
        return ", ".join([g['name'] for g in ast.literal_eval(x)])
    except:
        return "Unknown"

def parse_cast(x):
    try:
        return ", ".join([c['name'] for c in ast.literal_eval(x)[:3]])
    except:
        return "Unknown"

# --- clean metadata lookup ---
movies_meta = movies.copy()
movies_meta['genres_str'] = movies_meta['genres'].apply(parse_genres)
movies_meta['actors'] = movies_meta['id'].apply(
    lambda mid: parse_cast(cast_lookup.get(mid, '[]'))
)

movie_meta_map = {
    row.id: (row.original_title, row.genres_str, row.actors)
    for row in movies_meta.itertuples()
}

idx2movie = {v: k for k, v in movie_id_to_idx.items()}


def sample_candidates_for_user(user_id, train_df, n_candidates=25, seed=None):
    """Sample one positive + negatives once, shared by all models for this user."""
    user_rated = train_df[train_df['userId'] == user_id]['movieId'].unique()
    not_watched = list(all_movie_ids - set(user_rated))

    if len(user_rated) == 0 or len(not_watched) < n_candidates - 1:
        return None, None

    rng = random.Random(seed) if seed is not None else random
    watched_movie = rng.choice(list(user_rated))
    negative_sample = rng.sample(not_watched, n_candidates - 1)
    candidates = [watched_movie] + negative_sample
    rng.shuffle(candidates)
    return candidates, watched_movie


def top3_table(score_fn, user_id, train_df, candidates, n=3):
    """Return top-3 recommendation table for one user/model from shared candidates."""
    if not candidates:
        return None

    scores = score_fn(user_id, candidates)
    top_idx = np.argsort(scores)[::-1][:n]

    rows = []
    for idx in top_idx:
        movie_id = candidates[idx]
        title, genres, actors = movie_meta_map.get(
            movie_id, ("Unknown", "Unknown", "Unknown")
        )

        user_rating = train_df[
            (train_df['userId'] == user_id) &
            (train_df['movieId'] == movie_id)
        ]['rating']
        user_rating = user_rating.values[0] if len(user_rating) > 0 else "Not Rated"

        rows.append({
            "Movie": title,
            "Genres": genres,
            "Actors": actors,
            "Predicted Score": round(float(scores[idx]), 3),
            "User Rating": user_rating
        })

    return pd.DataFrame(rows)


# === Example run with shared candidates across all models ===
sample_user = test_users[12]
shared_candidates, watched_movie = sample_candidates_for_user(
    sample_user, ratings, n_candidates=25, seed=SEED
)

if shared_candidates is None:
    print(f"User {sample_user} does not have enough unrated movies to sample candidates.")
else:
    watched_title = movie_meta_map.get(watched_movie, ("Unknown",))[0]
    print(f"Positive movie in candidate set: {watched_title}")
    print(f"Candidate pool size: {len(shared_candidates)}")

    print("\nUSER-USER CF TOP-3")
    display(top3_table(score_user_user, sample_user, ratings, shared_candidates))

    print("\nITEM-ITEM (TF-IDF) TOP-3")
    display(top3_table(score_item_item_tfidf, sample_user, ratings, shared_candidates))

    print("\nGRADIENT BOOST TOP-3")
    display(top3_table(score_gradient_boost, sample_user, ratings, shared_candidates))

Positive movie in candidate set: Der Tunnel
Candidate pool size: 25

USER-USER CF TOP-3


,Movie,Genres,Actors,Predicted Score,User Rating
0,Der Tunnel,Science Fiction,"Paul Hartmann, Attila Hörbiger, Olly von Flint",1.5,5.0
1,Bandyta,Drama,"Til Schweiger, Polly Walker, Ida Jablonska",0.8,Not Rated
2,Captain Corelli's Mandolin,"Drama, History, Romance","Nicolas Cage, Penélope Cruz, John Hurt",0.7,Not Rated



ITEM-ITEM (TF-IDF) TOP-3


,Movie,Genres,Actors,Predicted Score,User Rating
0,Bandyta,Drama,"Til Schweiger, Polly Walker, Ida Jablonska",0.018,Not Rated
1,16 Blocks,"Action, Adventure, Crime, Thriller","Bruce Willis, Yasiin Bey, David Morse",0.015,Not Rated
2,Der Tunnel,Science Fiction,"Paul Hartmann, Attila Hörbiger, Olly von Flint",0.015,5.0



GRADIENT BOOST TOP-3


,Movie,Genres,Actors,Predicted Score,User Rating
0,Adaptation.,"Comedy, Crime, Drama","Nicolas Cage, Meryl Streep, Chris Cooper",1.0,Not Rated
1,Bandyta,Drama,"Til Schweiger, Polly Walker, Ida Jablonska",1.0,Not Rated
2,Captain Corelli's Mandolin,"Drama, History, Romance","Nicolas Cage, Penélope Cruz, John Hurt",1.0,Not Rated
